# Amazon Review Polarity — Data Exploration

A brief examination of the 3.6 million-row training set. The official test set is left untouched for final model evaluation.

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# REPRODUCIBILITY -- REPLACE THIS WITH YOUR OWN DATA PATH.
# Point it at the folder holding the Amazon Polarity CSVs.
# That folder must contain: train.csv and test.csv
# Everything else in this notebook is relative to the notebook
# directory (artifacts/, results_csv/) and needs no editing.
# ============================================================
DATA_DIR = Path("archive").resolve()
TRAIN_PATH = DATA_DIR / "train.csv"
COLUMNS = ["label", "title", "text"]
CHUNK_SIZE = 100_000

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {TRAIN_PATH}. Open the project folder before running the notebook."
    )

print("Training data:", TRAIN_PATH)

Training data: C:\Users\bsarv\Fake Desktop\amz sentiment analysis\archive\train.csv


## Preview

The file has three columns and no header. Label `1` is negative and label `2` is positive.

In [2]:
preview_pool = pd.read_csv(
    TRAIN_PATH,
    header=None,
    names=COLUMNS,
    keep_default_na=False,
    nrows=100_000,
)
preview = (
    preview_pool.groupby("label", group_keys=False)
                .sample(n=2, random_state=42)
                .sort_values("label")
)
preview.insert(1, "sentiment", preview["label"].map({1: "negative", 2: "positive"}))
display(preview)
del preview_pool

,label,sentiment,title,text
10627,1,negative,Stick with Swaddlers,"OK, this is the first review I have ever writt..."
2179,1,negative,Mindless and thoughtless,"Regardless of you religious beliefs, this book..."
95560,2,positive,LOVE IT,This pizza oven is great for cooking frozen pi...
76362,2,positive,I am usin this mouse as i write this.,This is a great and it made just my hand or I ...


## Training-set summary

The CSV is read in chunks so the entire file is never loaded into a DataFrame at once. This single pass collects class counts, blank fields, review lengths, and duplicate full rows.

In [3]:
class_counts = pd.Series(0, index=[1, 2], dtype="int64")
missing = {"title": {1: 0, 2: 0}, "text": {1: 0, 2: 0}}
review_lengths = {1: [], 2: []}
seen_hashes = set()
duplicate_rows = 0

start = time.perf_counter()

for chunk in pd.read_csv(
    TRAIN_PATH,
    header=None,
    names=COLUMNS,
    dtype={"label": "int8"},
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
):
    class_counts = class_counts.add(chunk["label"].value_counts(), fill_value=0)
    for label in (1, 2):
        part = chunk[chunk["label"].eq(label)]
        missing["title"][label] += int(part["title"].str.strip().eq("").sum())
        missing["text"][label] += int(part["text"].str.strip().eq("").sum())
        lengths = part["text"].str.count(r"\S+")
        review_lengths[label].append(lengths.astype("int32").to_numpy())

    hashes = pd.util.hash_pandas_object(chunk[COLUMNS], index=False)
    chunk_hashes = set(hashes)
    duplicate_rows += len(hashes) - len(chunk_hashes)
    duplicate_rows += len(chunk_hashes & seen_hashes)
    seen_hashes.update(chunk_hashes)

elapsed = time.perf_counter() - start
class_counts = class_counts.astype("int64")
unexpected_labels = sorted(set(class_counts.index) - {1, 2})
assert not unexpected_labels, f"Unexpected labels: {unexpected_labels}"
del seen_hashes

In [4]:
total_rows = int(class_counts.sum())

class_summary = pd.DataFrame({
    "sentiment": ["negative", "positive"],
    "rows": class_counts.reindex([1, 2]).to_numpy(),
}, index=pd.Index([1, 2], name="label"))
class_summary["percent"] = (class_summary["rows"] / total_rows * 100).round(2)

missing_summary = pd.DataFrame({
    "sentiment": ["negative", "positive"],
    "blank_titles": [missing["title"][1], missing["title"][2]],
    "blank_texts": [missing["text"][1], missing["text"][2]],
}, index=pd.Index([1, 2], name="label"))
missing_summary["blank_title_percent"] = (
    missing_summary["blank_titles"] / class_summary["rows"] * 100
).round(4)

length_summary = pd.DataFrame({
    "negative": pd.Series(np.concatenate(review_lengths[1])).describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ),
    "positive": pd.Series(np.concatenate(review_lengths[2])).describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ),
}).T[["count", "mean", "50%", "90%", "95%", "99%", "max"]].round(1)
length_summary.index.name = "sentiment"

print(f"Scanned {total_rows:,} rows in {elapsed:.1f} seconds.")
print("\nClass distribution")
display(class_summary)
print("Missing or blank fields")
display(missing_summary)
print("Review length in whitespace-separated words")
display(length_summary)
print(f"Duplicate full rows: {duplicate_rows:,} ({duplicate_rows / total_rows:.4%})")

Scanned 3,600,000 rows in 52.2 seconds.

Class distribution


,sentiment,rows,percent
label,,,
1,negative,1800000,50.0
2,positive,1800000,50.0


Missing or blank fields


,sentiment,blank_titles,blank_texts,blank_title_percent
label,,,,
1,negative,22,0,0.0012
2,positive,26,0,0.0014


Review length in whitespace-separated words


,count,mean,50%,90%,95%,99%,max
sentiment,,,,,,,
negative,1800000.0,77.1,70.0,141.0,157.0,177.0,254.0
positive,1800000.0,71.2,62.0,136.0,154.0,175.0,219.0


Duplicate full rows: 0 (0.0000%)
